In [1]:
import sys
sys.path.insert(0, "..")

import mne
from pathlib import Path
from src.preprocessing.loader import load_raw
from src.preprocessing.filter import apply_filters
from src.preprocessing.epoching import make_epochs,summarize_epochs
from src.visualization.plot import plot_erp, plot_topomap
from pipeline import load_config

cfg = load_config("../config.yaml")
print(f"MNE version: {mne.__version__}")


MNE version: 1.12.1


In [2]:
USE_SAMPLE_DATA = True  # flip to False (and set the path below) to run against a local recording

if USE_SAMPLE_DATA:
    sample_path = mne.datasets.sample.data_path()
    raw_path = sample_path / "MEG" / "sample" / "sample_audvis_raw.fif"
else:
    raw_path = Path(cfg["paths"]["raw_data"]) / "your_recording.fif"

raw = load_raw(str(raw_path))
raw_eeg = raw.copy().pick_types(eeg=True, meg=False, stim=True, eog=False)
bp = cfg["preprocessing"]["bandpass"]
raw_filtered = apply_filters(raw_eeg, l_freq=bp["l_freq"], h_freq=bp["h_freq"], notch_freq=cfg["preprocessing"]["notch_freq"])

Opening raw data file C:\Users\ke725\mne_data\MNE-sample-data\MEG\sample\sample_audvis_raw.fif...


    Read a total of 3 projection items:


        PCA-v1 (1 x 102)  idle


        PCA-v2 (1 x 102)  idle


        PCA-v3 (1 x 102)  idle


...

- Lower passband edge: 1.00


- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)


- Upper passband edge: 40.00 Hz


- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)


- Filter length: 1983 samples (3.302 s)


In [3]:
pp = cfg["preprocessing"]
epochs = make_epochs(raw_filtered, tmin=pp["epoch_tmin"], tmax=pp["epoch_tmax"], baseline=(None, 0))
summarize_epochs(epochs)

Finding events on: STI 014


320 events found on stim channel STI 014


Event IDs: [ 1  2  3  4  5 32]


Not setting metadata


320 matching events found


...

Applying baseline correction (mode: mean)


0 projection items activated


Using data from preloaded Raw for 320 events and 601 original time points ...


0 bad epochs dropped


Events found : 320
Epochs kept  : 320
Epochs dropped : 0


In [4]:
epochs_baselined = make_epochs(raw_filtered, tmin=pp["epoch_tmin"], tmax=pp["epoch_tmax"], baseline=(None, 0))
epochs_no_baseline = make_epochs(raw_filtered, tmin=pp["epoch_tmin"], tmax=pp["epoch_tmax"], baseline=None)
evoked_baselined = epochs_baselined.average()
evoked_no_baseline = epochs_no_baseline.average()
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
evoked_no_baseline.plot(axes=axes[0], show=False)
axes[0].set_title("No baseline correction")
evoked_baselined.plot(axes=axes[1], show=False)
axes[1].set_title("Baseline corrected (config.yaml default)")
plt.tight_layout()
plt.show()

Finding events on: STI 014


320 events found on stim channel STI 014


Event IDs: [ 1  2  3  4  5 32]


Not setting metadata


320 matching events found


...

No baseline correction applied


0 projection items activated


Using data from preloaded Raw for 320 events and 601 original time points ...


0 bad epochs dropped


C:\Users\ke725\AppData\Local\Temp\ipykernel_38404\2920540242.py:11: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()
C:\Users\ke725\AppData\Local\Temp\ipykernel_38404\2920540242.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [5]:
plot_topomap(evoked_baselined, times=[0.0, 0.1, 0.3, 0.5])

C:\Users\ke725\Documents\eeg-project\notebooks\..\src\visualization\plot.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**What I Observed in M3**

**Epoch counts:** 320 events found, 320 epochs kept, 0 dropped — every epoch fit
cleanly inside the recording bounds and the -0.2 to 0.8 s window.

**Baseline correction effect:**
- The two Evoked plots have the same waveform shape and the same peak timing
  (deflections at ~0.1 s and ~0.18 s) — baseline correction didn't change the
  ERP morphology at all.
- What it did change is a vertical offset: without correction, the
  pre-stimulus window sits around +1 to +2 µV instead of at 0; after
  correction, the pre-stim period is centered at 0 µV as expected, and the
  whole trace shifts down accordingly (post-stimulus range extends to about
  -6 µV instead of -5 µV).
- The effect was real but modest, not dramatic. That suggests M2's 1-40 Hz
  bandpass filter already removed most of the slow drift, leaving only a
  small residual DC-like offset for baseline correction to clean up.

**Topomap:**
- Yes, the scalp pattern shifts a lot across the four time points. At 0.0 s
  (stimulus onset) the map is nearly flat, close to 0 µV everywhere, as
  expected right after baseline correction.
- 0.1 s is the most "focused" — a strong, clearly bipolar pattern: a large
  negative (blue) region over central/frontal cortex paired with a positive
  (red) region at the posterior pole, reaching the full ±6 µV range. This
  lines up exactly with the early ERP peak seen in the Evoked plot.
- 0.3 s and 0.5 s are both much more "spread out" and washed out — weaker,
  more diffuse patterns with low magnitude and no clear focal structure,
  consistent with the response decaying after the early peak.

**Why Each Upcoming Module Now Makes Sense**

- **M4 artifact rejection:** the epoch-drop count in summarize_epochs only
  catches trials that don't fit the time window — it says nothing about
  trials that fit fine but contain a huge blink or muscle spike. That's
  M4's job, using the same Epochs object built here.
- **M5 ICA:** ICA runs on this same epoched, baseline-corrected data to
  separate brain sources from eye/muscle sources statistically.
- **M6 ERP:** epochs.average() (used here to build evoked_baselined) is
  exactly the operation M6's ERP analysis is built around — this notebook
  already previewed it.
